# BioTutor — LoRA fine-tuning (Google Colab)

Fine-tunes **TinyLlama-1.1B-Chat** into a bioinformatics tutor using **LoRA** (fp16, no
4-bit / no bitsandbytes). We train **only on the assistant's answer** so the model learns
to reply and then **stop** cleanly.

**Before you start:** enable the GPU → `Runtime` ▸ `Change runtime type` ▸ **T4 GPU** ▸ Save.

Run the cells in order, top to bottom. If you restart the session, start again from cell 1.

## 1. Install dependencies (pinned)

In [ ]:
# No bitsandbytes: we train in fp16. Ignore the red 'dependency conflicts' warnings
# about numpy/jax/opencv etc. - they refer to unrelated Colab packages we do not use.
%pip install -q -U \
    transformers==4.44.2 \
    trl==0.9.6 \
    peft==0.12.0 \
    datasets==2.20.0 \
    accelerate==0.33.0

## 2. Upload the training data
Run the cell, then choose **train.jsonl** and **val.jsonl** (produced by `build_dataset.py`).
Outside Colab, just place those two files next to the notebook.

In [ ]:
import os
try:
    from google.colab import files          # only exists on Google Colab
    print("Select train.jsonl and val.jsonl from your computer...")
    files.upload()
except ModuleNotFoundError:
    missing = [f for f in ("train.jsonl", "val.jsonl") if not os.path.exists(f)]
    assert not missing, (
        f"Not on Colab and missing {missing}. Either run this notebook on "
        "Google Colab, or put train.jsonl and val.jsonl next to it."
    )
    print("Running outside Colab - using local train.jsonl / val.jsonl")

## 3. Imports and GPU check

In [ ]:
import torch
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, TrainingArguments,
                          Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 4. Load the dataset

In [ ]:
data = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
print(data)
print("\nExample instruction:", data["train"][0]["instruction"])
print("Example response   :", data["train"][0]["response"])

## 5. Load TinyLlama (fp16)
TinyLlama-1.1B is only ~2.2 GB in fp16, so it fits comfortably on a free T4 without
quantization.

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # TinyLlama has no dedicated pad token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False              # required during training
print("Base model loaded in fp16.")

## 6. Attach LoRA adapters
We freeze the base model and add small trainable low-rank adapters to the attention and
MLP projections.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()          # should show a small % of trainable params

## 7. Prepare the data (train only on the answer)
For every example we build the full chat string, tokenize it, and set the label of every
**prompt** token to `-100`. That means the loss is computed **only on the assistant's
answer and its EOS token**, which is what teaches the model to answer and then STOP.

In [ ]:
SYSTEM = ("You are BioTutor, an expert tutor in bioinformatics, biology, and machine "
          "learning. You give thorough, detailed explanations with concrete examples, and "
          "you stay detailed even when the question is short.")
MAXLEN = 1024   # longer, so detailed answers are not truncated

def preprocess(ex):
    prompt = (f"<|system|>\n{SYSTEM}</s>\n"
              f"<|user|>\n{ex['instruction']}</s>\n"
              f"<|assistant|>\n")
    full = prompt + ex["response"] + tokenizer.eos_token
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    full_ids   = tokenizer(full,   add_special_tokens=True)["input_ids"][:MAXLEN]
    labels = list(full_ids)
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100                    # mask the prompt
    return {"input_ids": full_ids,
            "attention_mask": [1] * len(full_ids),
            "labels": labels}

train_tok = data["train"].map(preprocess, remove_columns=data["train"].column_names)
val_tok   = data["validation"].map(preprocess, remove_columns=data["validation"].column_names)
print("Tokenized. Example length:", len(train_tok[0]["input_ids"]),
      "| answer tokens:", sum(1 for x in train_tok[0]["labels"] if x != -100))

## 8. Training configuration and Trainer
Small dataset, so a modest effective batch (8) and 12 epochs. Watch the validation loss:
if it rises while training loss keeps falling, lower `num_train_epochs`.

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

training_args = TrainingArguments(
    output_dir="biotutor-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,          # effective batch size = 8
    num_train_epochs=12,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

## 9. Train
The training loss should start around 2-3 and fall. **It must NOT be 0.0** — if it is,
stop and check cell 7.

In [ ]:
trainer.train()

## 10. Quick test
Ask the fine-tuned model some questions. It should answer and stop on its own.

In [ ]:
SYSTEM = ("You are BioTutor, an expert tutor in bioinformatics, biology, and machine "
          "learning. You give thorough, detailed explanations with concrete examples, and "
          "you stay detailed even when the question is short.")
model.config.use_cache = True   # faster generation

def ask(question, max_new_tokens=400):
    prompt = f"<|system|>\n{SYSTEM}</s>\n<|user|>\n{question}</s>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,                       # greedy: focused and stops sooner
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    answer = text.split("<|")[0].strip()       # safety guard against any leftover markers
    print("Q:", question)
    print("A:", answer, "\n")

ask("What is GC content?")
ask("What is the difference between a transition and a transversion?")
ask("What is overfitting in machine learning?")
ask("How do I detect mutations in the Sequence Toolkit?")

## 11. Save and download the adapter
The LoRA adapter is small (a few MB). We zip it and download it — this is what the backend
will load to serve the model.

In [ ]:
model.save_pretrained("biotutor-lora")
tokenizer.save_pretrained("biotutor-lora")

!zip -r biotutor-lora.zip biotutor-lora >/dev/null
from google.colab import files
files.download("biotutor-lora.zip")
print("Done. Keep biotutor-lora.zip for the serving step.")

## Next step
Unzip `biotutor-lora.zip` into the project's `backend/` folder. The FastAPI backend
(next step) will load the base model plus this adapter and expose a chat endpoint.